# my-travel-world API — Tool Reference for LLM Travel Planning Agents

This notebook demonstrates every major `my-travel-world` REST endpoint as a ready-to-use Python tool function.
Each function wraps one endpoint with **all** keyword arguments explicit, so it can be dropped directly into
an LLM agent's tool registry (Anthropic, OpenAI function-calling, LangChain, etc.).

**Prerequisites**
- The API server is running locally (`python scripts/run_api.py`)
- At least one world has been generated (`python scripts/generate_world.py --seed 42`)
- `httpx` is installed (it is a project dependency)

## 0 · Setup

In [ ]:
import httpx, json, pprint

BASE_URL = "http://localhost:8000"   # change if your server runs elsewhere

# Thin synchronous client — all tool functions share this
client = httpx.Client(base_url=BASE_URL, timeout=30.0)

def _get(path: str, **params) -> dict | list:
    """GET helper — strips None params so optional args are omitted cleanly."""
    r = client.get(path, params={k: v for k, v in params.items() if v is not None})
    r.raise_for_status()
    return r.json()

def _post(path: str, **body) -> dict:
    """POST helper — sends JSON body, strips None values."""
    r = client.post(path, json={k: v for k, v in body.items() if v is not None})
    r.raise_for_status()
    return r.json()

def _delete(path: str) -> dict:
    r = client.delete(path)
    r.raise_for_status()
    return r.json()

pp = pprint.PrettyPrinter(indent=2, depth=4)
print("Client ready.", _get("/health"))

---
## 1 · World Management

Discover available worlds, load one as the active context, and inspect its geography.
All subsequent tool calls operate against the **active world**.

In [ ]:
def list_worlds() -> list[dict]:
    """Return all generated world instances with metadata."""
    return _get("/world/")

def generate_world(
    *,
    seed: int,
    world_id: str | None = None,
) -> dict:
    """
    Procedurally generate a new travel world.

    Args:
        seed:     RNG seed — same seed always produces the same world.
        world_id: Optional stable name; auto-generated (UUID) if omitted.
    Returns:
        {world_id, seed, summary}
    """
    return _post("/world/generate", seed=seed, world_id=world_id)

def get_world(
    *,
    world_id: str,
) -> dict:
    """Fetch metadata and layer summary for a world."""
    return _get(f"/world/{world_id}")

def load_world(
    *,
    world_id: str,
) -> dict:
    """
    Set a world as **active** for all subsequent API calls.
    Must be called before any search/booking tool.
    """
    return _post(f"/world/{world_id}/load")

def list_cities(
    *,
    world_id: str,
) -> list[dict]:
    """List every city in a world — use city_id values in search tools."""
    return _get(f"/world/{world_id}/cities")

def list_districts(
    *,
    world_id: str,
    city_id: str | None = None,   # filter to one city; None → all districts
) -> list[dict]:
    """List districts, optionally scoped to a single city."""
    return _get(f"/world/{world_id}/districts", city_id=city_id)

def get_map_data(
    *,
    world_id: str,
) -> dict:
    """Return all geographic data (cities, districts, locations, flight routes) for map rendering."""
    return _get(f"/world/{world_id}/map_data")

In [ ]:
# ── Example: discover worlds and load one ────────────────────────────────────
worlds = list_worlds()
print(f"Found {len(worlds)} world(s)")

# Pick the first available world (or generate one if none exist)
if worlds:
    WORLD_ID = worlds[0]["world_id"]
else:
    result = generate_world(seed=42, world_id="demo-world")
    WORLD_ID = result["world_id"]

print("Using world:", WORLD_ID)
load_result = load_world(world_id=WORLD_ID)
print(load_result["message"])

In [ ]:
# ── Example: inspect cities and pick origin / destination ────────────────────
cities = list_cities(world_id=WORLD_ID)
print(f"{len(cities)} cities:")
for c in cities[:5]:  # preview first 5
    print(f"  {c['city_id']:20s}  {c['name']}")

# Convenience references used throughout this notebook
ORIGIN_CITY      = cities[0]["city_id"]
DEST_CITY        = cities[1]["city_id"]
print(f"\nOrigin: {ORIGIN_CITY}   Destination: {DEST_CITY}")

---
## 2 · Session Management

A **session** is the agent's working memory: it holds trip preferences, the evolving trip plan,
chat history, and an interaction log used for evaluation.  Create one session per conversation.

In [ ]:
def create_session(
    *,
    world_id: str,
) -> dict:
    """
    Create a new agent session tied to a world.

    Returns:
        {session_id, world_id} — save session_id for every subsequent call.
    """
    return _post("/session/", world_id=world_id)

def get_session(
    *,
    session_id: str,
) -> dict:
    """
    Fetch full session state — the primary **bootstrap** endpoint.
    Call this first to hydrate the agent with current preferences, trip plan, and history.
    """
    return _get(f"/session/{session_id}")

def update_preferences(
    *,
    session_id: str,
    origin_city_id: str | None = None,
    destination_city_ids: list[str] | None = None,
    departure_date: str | None = None,       # "YYYY-MM-DD"
    return_date: str | None = None,          # "YYYY-MM-DD"
    budget_total: float | None = None,
    group_size: int | None = None,
    travel_style: str | None = None,         # e.g. "adventure", "luxury", "budget"
    pace: str | None = None,                 # e.g. "relaxed", "packed"
    preferred_transport: list[str] | None = None,  # ["flight", "train", "bus"]
) -> dict:
    """
    Set or update traveller preferences.  Only the fields you supply are changed;
    all others remain as-is.
    """
    prefs = dict(
        origin_city_id=origin_city_id,
        destination_city_ids=destination_city_ids,
        departure_date=departure_date,
        return_date=return_date,
        budget_total=budget_total,
        group_size=group_size,
        travel_style=travel_style,
        pace=pace,
        preferred_transport=preferred_transport,
    )
    return client.put(
        f"/session/{session_id}/preferences",
        json={k: v for k, v in prefs.items() if v is not None},
    ).json()

def post_chat_message(
    *,
    session_id: str,
    role: str,      # "user" or "assistant"
    content: str,
) -> dict:
    """Append a message to the session's chat history (visible in the UI)."""
    return _post(f"/session/{session_id}/chat_message", role=role, content=content)

def get_interaction_log(
    *,
    session_id: str,
) -> list[dict]:
    """Export the full interaction log — used for offline agent evaluation."""
    return _get(f"/session/{session_id}/interaction_log")

In [ ]:
# ── Example: create a session and set traveller preferences ──────────────────
session = create_session(world_id=WORLD_ID)
SESSION_ID = session["session_id"]
print("Session:", SESSION_ID)

prefs = update_preferences(
    session_id=SESSION_ID,
    origin_city_id=ORIGIN_CITY,
    destination_city_ids=[DEST_CITY],
    departure_date="2025-07-15",
    return_date="2025-07-22",
    budget_total=3000.0,
    group_size=2,
    travel_style="adventure",
    pace="relaxed",
    preferred_transport=["flight", "train"],
)
pp.pprint(prefs["preferences"])

---
## 3 · Flights

Search for flights between cities and retrieve route or detail information.

In [ ]:
def search_flights(
    *,
    origin_city_id: str,
    destination_city_id: str,
    departure_date: str,           # "YYYY-MM-DD"
    passengers: int = 1,
    cabin_class: str | None = None,  # "economy" | "business" | "first"
    session_id: str | None = None,   # enables interaction logging
) -> dict:
    """
    Search available flights.

    Returns:
        {flights: list[FlightResult], world_id, search_date, total_results}

    Key FlightResult fields:
        flight_id, airline, flight_number, departure_datetime, arrival_datetime,
        price_per_person, total_price, duration_min, seats_available, baggage_included
    """
    return _get(
        "/flights/search",
        origin_city_id=origin_city_id,
        destination_city_id=destination_city_id,
        departure_date=departure_date,
        passengers=passengers,
        cabin_class=cabin_class,
        session_id=session_id,
    )

def get_flight(
    *,
    flight_id: str,
) -> dict:
    """Retrieve full details for a specific flight."""
    return _get(f"/flights/{flight_id}")

def list_flight_routes() -> list[dict]:
    """List all available origin→destination route pairs in the active world."""
    return _get("/flights/routes")

In [ ]:
# ── Example A: economy search for 2 passengers ───────────────────────────────
result_eco = search_flights(
    origin_city_id=ORIGIN_CITY,
    destination_city_id=DEST_CITY,
    departure_date="2025-07-15",
    passengers=2,
    cabin_class="economy",
    session_id=SESSION_ID,
)
print(f"Economy — {result_eco['total_results']} flight(s) found")
for f in result_eco["flights"][:2]:
    print(f"  {f['airline']} {f['flight_number']}  "
          f"departs {f['departure_datetime']}  "
          f"${f['total_price']:.0f} total  "
          f"{f['duration_min']}min")

In [ ]:
# ── Example B: business class, single passenger ───────────────────────────────
result_biz = search_flights(
    origin_city_id=ORIGIN_CITY,
    destination_city_id=DEST_CITY,
    departure_date="2025-07-15",
    passengers=1,
    cabin_class="business",
    session_id=SESSION_ID,
)
print(f"Business — {result_biz['total_results']} flight(s) found")
for f in result_biz["flights"][:2]:
    print(f"  {f['airline']} {f['flight_number']}  ${f['price_per_person']:.0f}/person")

In [ ]:
# ── Example C: no cabin filter (all classes) — inspect a specific flight ─────
result_all = search_flights(
    origin_city_id=ORIGIN_CITY,
    destination_city_id=DEST_CITY,
    departure_date="2025-07-15",
    passengers=1,
    cabin_class=None,      # no filter
    session_id=SESSION_ID,
)
if result_all["flights"]:
    fid = result_all["flights"][0]["flight_id"]
    detail = get_flight(flight_id=fid)
    pp.pprint(detail)

# save for trip-plan demo later
FLIGHT_ID = result_all["flights"][0]["flight_id"] if result_all["flights"] else None

---
## 4 · Hotels

Search, compare, check availability, and book hotels.

In [ ]:
def search_hotels(
    *,
    city_id: str,
    check_in: str,                          # "YYYY-MM-DD"
    check_out: str,                         # "YYYY-MM-DD"
    guests: int = 1,
    max_price_per_night: float | None = None,
    min_stars: int | None = None,           # 1–5
    required_amenities: str | None = None,  # comma-separated: "pool,gym,wifi"
    session_id: str | None = None,
) -> dict:
    """
    Search available hotels.

    Returns:
        {hotels: list[dict], world_id, total_results}

    Key hotel fields:
        hotel_id, name, stars, price_per_night, amenities, district_id,
        average_rating, total_reviews, available
    """
    return _get(
        "/hotels/search",
        city_id=city_id,
        check_in=check_in,
        check_out=check_out,
        guests=guests,
        max_price_per_night=max_price_per_night,
        min_stars=min_stars,
        required_amenities=required_amenities,
        session_id=session_id,
    )

def get_hotel(
    *,
    hotel_id: str,
) -> dict:
    """Full details + reviews for a hotel."""
    return _get(f"/hotels/{hotel_id}")

def get_hotel_availability(
    *,
    hotel_id: str,
    check_in: str,   # "YYYY-MM-DD"
    check_out: str,  # "YYYY-MM-DD"
) -> list[dict]:
    """Per-night availability calendar for a hotel over the requested range."""
    return _get(
        f"/hotels/{hotel_id}/availability",
        check_in=check_in,
        check_out=check_out,
    )

def compare_hotels(
    *,
    hotel_ids: list[str],  # compare up to ~5 hotels
    check_in: str,
    check_out: str,
) -> list[dict]:
    """Side-by-side comparison of multiple hotels."""
    return _get(
        "/hotels/compare",
        hotel_ids=",".join(hotel_ids),  # API expects comma-separated string
        check_in=check_in,
        check_out=check_out,
    )

def book_hotel(
    *,
    hotel_id: str,
    check_in: str,
    check_out: str,
    session_id: str,
) -> dict:
    """
    Book a hotel room and add it to the session trip plan.

    Returns:
        {booking_id, hotel_name, check_in, check_out, num_nights, total_cost, status}
    """
    return _post(
        f"/hotels/{hotel_id}/book",
        check_in=check_in,
        check_out=check_out,
        session_id=session_id,
    )

In [ ]:
# ── Example A: broad search — all hotels, any price ──────────────────────────
hotels_all = search_hotels(
    city_id=DEST_CITY,
    check_in="2025-07-15",
    check_out="2025-07-22",
    guests=2,
    max_price_per_night=None,
    min_stars=None,
    required_amenities=None,
    session_id=SESSION_ID,
)
print(f"{hotels_all['total_results']} hotels found")
for h in hotels_all["hotels"][:3]:
    print(f"  {h['name']:30s}  ★{h['stars']}  ${h['price_per_night']:.0f}/night")

In [ ]:
# ── Example B: luxury filter — 4+ stars, pool & gym ─────────────────────────
hotels_luxury = search_hotels(
    city_id=DEST_CITY,
    check_in="2025-07-15",
    check_out="2025-07-22",
    guests=2,
    max_price_per_night=500.0,
    min_stars=4,
    required_amenities="pool,gym",
    session_id=SESSION_ID,
)
print(f"Luxury filter → {hotels_luxury['total_results']} result(s)")
for h in hotels_luxury["hotels"][:3]:
    print(f"  {h['name']:30s}  ★{h['stars']}  ${h['price_per_night']:.0f}/night")

# save for comparison + booking demos
HOTEL_IDS = [h["hotel_id"] for h in hotels_all["hotels"][:3]]
HOTEL_ID  = HOTEL_IDS[0] if HOTEL_IDS else None

In [ ]:
# ── Example C: compare top-3 hotels side-by-side ────────────────────────────
if len(HOTEL_IDS) >= 2:
    comparison = compare_hotels(
        hotel_ids=HOTEL_IDS,
        check_in="2025-07-15",
        check_out="2025-07-22",
    )
    for h in comparison:
        print(f"{h['name']:30s}  ★{h['stars']}  ${h['price_per_night']:.0f}/night  "
              f"rating: {h.get('average_rating', 'N/A')}")

In [ ]:
# ── Example D: per-night availability calendar ───────────────────────────────
if HOTEL_ID:
    avail = get_hotel_availability(
        hotel_id=HOTEL_ID,
        check_in="2025-07-15",
        check_out="2025-07-22",
    )
    for night in avail:
        status = "✓" if night.get("available") else "✗"
        print(f"  {night['date']}  {status}  ${night.get('price', '?'):.0f}")

---
## 5 · Attractions

Discover sights, estimate crowd levels, and find what's nearby a given location.

In [ ]:
def search_attractions(
    *,
    city_id: str,
    category: str | None = None,        # e.g. "museum", "park", "landmark"
    district_id: str | None = None,
    max_ticket_price: float | None = None,
    free_only: bool = False,
    session_id: str | None = None,
) -> list[dict]:
    """
    Search attractions with optional category / location / price filters.

    Key result fields:
        attraction_id, name, category, ticket_price, rating,
        typical_duration_min, description
    """
    return _get(
        "/attractions/search",
        city_id=city_id,
        category=category,
        district_id=district_id,
        max_ticket_price=max_ticket_price,
        free_only=free_only,
        session_id=session_id,
    )

def get_attraction(
    *,
    attraction_id: str,
) -> dict:
    """Full details for a specific attraction."""
    return _get(f"/attractions/{attraction_id}")

def get_nearby_attractions(
    *,
    location_id: str,
    radius_km: float = 1.0,
    location_types: str | None = None,  # comma-separated: "attraction,restaurant"
) -> list[dict]:
    """Find points of interest within `radius_km` of a named location."""
    return _get(
        "/attractions/nearby",
        location_id=location_id,
        radius_km=radius_km,
        location_types=location_types,
    )

def get_attraction_crowding(
    *,
    attraction_id: str,
    visit_datetime: str,  # ISO 8601: "2025-07-16T10:00:00"
) -> dict:
    """
    Forecast crowd level for a given visit time.

    Useful for recommending off-peak visit windows.
    Returns: {crowding_level: str, estimated_wait_min: int, ...}
    """
    return _get(
        f"/attractions/{attraction_id}/crowding",
        visit_datetime=visit_datetime,
    )

In [ ]:
# ── Example A: all free attractions ──────────────────────────────────────────
free_attr = search_attractions(
    city_id=DEST_CITY,
    category=None,
    district_id=None,
    max_ticket_price=None,
    free_only=True,
    session_id=SESSION_ID,
)
print(f"{len(free_attr)} free attraction(s):")
for a in free_attr[:4]:
    print(f"  {a['name']:30s}  {a.get('category', '')}")

In [ ]:
# ── Example B: museums under $25 ─────────────────────────────────────────────
museums = search_attractions(
    city_id=DEST_CITY,
    category="museum",
    district_id=None,
    max_ticket_price=25.0,
    free_only=False,
    session_id=SESSION_ID,
)
print(f"{len(museums)} museum(s) ≤ $25:")
for a in museums[:4]:
    print(f"  {a['name']:30s}  ${a.get('ticket_price', 0):.0f}  ★{a.get('rating', '?')}")

# save for crowding demo
ATTRACTION_ID = museums[0]["attraction_id"] if museums else (
    free_attr[0]["attraction_id"] if free_attr else None
)

In [ ]:
# ── Example C: crowding forecast at different times of day ───────────────────
if ATTRACTION_ID:
    for hour in ["09:00:00", "13:00:00", "17:00:00"]:
        crowd = get_attraction_crowding(
            attraction_id=ATTRACTION_ID,
            visit_datetime=f"2025-07-16T{hour}",
        )
        print(f"  {hour}  → {crowd.get('crowding_level', '?'):10s}  "
              f"est. wait: {crowd.get('estimated_wait_min', '?')} min")

---
## 6 · Events

Browse local events, view a monthly calendar, and book tickets.

In [ ]:
def search_events(
    *,
    city_id: str,
    start_date: str | None = None,   # "YYYY-MM-DD" — inclusive
    end_date: str | None = None,     # "YYYY-MM-DD" — inclusive
    category: str | None = None,     # e.g. "concert", "festival", "sports"
    max_price: float | None = None,
    session_id: str | None = None,
) -> list[dict]:
    """
    Search events with optional date-range, category, and price filters.

    Key result fields:
        event_id, name, category, start_datetime, end_datetime,
        price_per_ticket, tickets_remaining, venue_name
    """
    return _get(
        "/events/search",
        city_id=city_id,
        start_date=start_date,
        end_date=end_date,
        category=category,
        max_price=max_price,
        session_id=session_id,
    )

def get_event_calendar(
    *,
    city_id: str,
    year: int,
    month: int,  # 1–12
) -> dict:
    """
    Return a month's events grouped by day: {"YYYY-MM-DD": [event, ...]}
    Useful for presenting an overview before the user picks a travel window.
    """
    return _get(
        "/events/calendar",
        city_id=city_id,
        year=year,
        month=month,
    )

def get_event(
    *,
    event_id: str,
) -> dict:
    """Full details for a specific event."""
    return _get(f"/events/{event_id}")

def book_event(
    *,
    event_id: str,
    quantity: int = 1,
    session_id: str = "",
) -> dict:
    """
    Purchase event tickets and record them in the session trip plan.

    Raises HTTP 409 if insufficient tickets remain.
    Returns:
        {event_id, event_name, quantity, total_cost, start_datetime, tickets_remaining}
    """
    return _post(
        "/events/book",
        event_id=event_id,
        quantity=quantity,
        session_id=session_id,
    )

In [ ]:
# ── Example A: all events during the trip window ─────────────────────────────
events_all = search_events(
    city_id=DEST_CITY,
    start_date="2025-07-15",
    end_date="2025-07-22",
    category=None,
    max_price=None,
    session_id=SESSION_ID,
)
print(f"{len(events_all)} event(s) during trip window")
for e in events_all[:4]:
    print(f"  {e['name']:35s}  {e['start_datetime'][:10]}  "
          f"${e.get('price_per_ticket', 0):.0f}/ticket")

In [ ]:
# ── Example B: concerts only, budget ≤ $50 ───────────────────────────────────
concerts = search_events(
    city_id=DEST_CITY,
    start_date="2025-07-01",
    end_date="2025-07-31",
    category="concert",
    max_price=50.0,
    session_id=SESSION_ID,
)
print(f"{len(concerts)} concert(s) ≤ $50")
for e in concerts[:3]:
    print(f"  {e['name']:35s}  {e['start_datetime']}")

EVENT_ID = (events_all or concerts)[0]["event_id"] if (events_all or concerts) else None

In [ ]:
# ── Example C: monthly calendar view ─────────────────────────────────────────
calendar = get_event_calendar(
    city_id=DEST_CITY,
    year=2025,
    month=7,
)
busy_days = {day: len(evs) for day, evs in calendar.items() if evs}
print("Days with events:", sorted(busy_days.items())[:8])

---
## 7 · Restaurants

Find dining options by cuisine, price point, or district.

In [ ]:
def search_restaurants(
    *,
    city_id: str,
    cuisine: str | None = None,                 # e.g. "italian", "sushi", "local"
    max_avg_spend: float | None = None,          # average spend per person
    reservation_required: bool | None = None,    # None → no filter
    district_id: str | None = None,
    session_id: str | None = None,
) -> dict:
    """
    Search restaurants.

    Returns:
        {restaurants: list[dict], total_results}

    Key result fields:
        restaurant_id, name, cuisine, avg_spend_per_person,
        average_rating, reservation_required, district_id
    """
    return _get(
        "/restaurants/search",
        city_id=city_id,
        cuisine=cuisine,
        max_avg_spend=max_avg_spend,
        reservation_required=reservation_required,
        district_id=district_id,
        session_id=session_id,
    )

def get_restaurant(
    *,
    restaurant_id: str,
) -> dict:
    """Full details and reviews for a specific restaurant."""
    return _get(f"/restaurants/{restaurant_id}")

In [ ]:
# ── Example A: all restaurants in destination city ───────────────────────────
rests_all = search_restaurants(
    city_id=DEST_CITY,
    cuisine=None,
    max_avg_spend=None,
    reservation_required=None,
    district_id=None,
    session_id=SESSION_ID,
)
print(f"{rests_all['total_results']} restaurant(s) found")
for r in rests_all["restaurants"][:4]:
    print(f"  {r['name']:30s}  {r.get('cuisine', ''):12s}  "
          f"${r.get('avg_spend_per_person', 0):.0f}/person  ★{r.get('average_rating', '?')}")

In [ ]:
# ── Example B: walk-in friendly Italian, ≤ $40/person ────────────────────────
rests_filtered = search_restaurants(
    city_id=DEST_CITY,
    cuisine="italian",
    max_avg_spend=40.0,
    reservation_required=False,
    district_id=None,
    session_id=SESSION_ID,
)
print(f"{rests_filtered['total_results']} walk-in Italian ≤ $40")
for r in rests_filtered["restaurants"][:3]:
    print(f"  {r['name']}")

---
## 8 · Routing

Plan multi-modal routes between locations, compare transport options, and search by proximity.

In [ ]:
def plan_route(
    *,
    origin_location_id: str,
    destination_location_id: str,
    departure_datetime: str,               # ISO 8601: "2025-07-16T09:00:00"
    modes: str | None = None,              # comma-separated: "walking,taxi,subway"
    optimize_for: str = "time",            # "time" | "cost" | "balanced"
    session_id: str | None = None,
) -> dict:
    """
    Plan multi-modal routes between any two locations.

    Returns:
        {routes: list[dict], world_id, origin_name, destination_name}

    Key route fields:
        mode, duration_min, cost, legs, congestion_applied
    """
    return _get(
        "/routing/plan",
        origin_location_id=origin_location_id,
        destination_location_id=destination_location_id,
        departure_datetime=departure_datetime,
        modes=modes,
        optimize_for=optimize_for,
        session_id=session_id,
    )

def compare_transport_modes(
    *,
    origin_location_id: str,
    destination_location_id: str,
    departure_datetime: str,   # ISO 8601
) -> dict:
    """Compare all available transport modes side-by-side for the same journey."""
    return _get(
        "/routing/compare",
        origin_location_id=origin_location_id,
        destination_location_id=destination_location_id,
        departure_datetime=departure_datetime,
    )

def get_travel_time(
    *,
    origin_id: str,
    destination_id: str,
    mode: str,                 # "walking" | "taxi" | "subway" | "bus" | ...
    departure_datetime: str,   # ISO 8601
) -> dict:
    """
    Get travel time and cost for a specific mode and departure time.

    Returns:
        {duration_min, cost, congestion_applied, available}
    """
    return _get(
        "/routing/time",
        origin_id=origin_id,
        destination_id=destination_id,
        mode=mode,
        departure_datetime=departure_datetime,
    )

def proximity_search(
    *,
    lat: float,
    lon: float,
    top_n: int = 10,
    sort_by: str = "distance",           # "distance" | "travel_time"
    location_type: str | None = None,    # "attraction" | "hotel" | "restaurant" | ...
    city_id: str | None = None,          # restrict to one city
    mode: str = "walking",
) -> dict:
    """
    Find the nearest locations to a coordinate.

    Returns:
        {results: list[dict], count}
    """
    return _get(
        "/routing/nearby",
        lat=lat,
        lon=lon,
        top_n=top_n,
        sort_by=sort_by,
        location_type=location_type,
        city_id=city_id,
        mode=mode,
    )

In [ ]:
# Grab two location IDs from map data to use as routing endpoints
map_data  = get_map_data(world_id=WORLD_ID)
locations = map_data.get("locations", [])

dest_locs = [l for l in locations if l.get("city_id") == DEST_CITY]
ORIGIN_LOC = dest_locs[0]["location_id"] if len(dest_locs) > 0 else None
DEST_LOC   = dest_locs[1]["location_id"] if len(dest_locs) > 1 else None
print("Routing from", ORIGIN_LOC, "→", DEST_LOC)

In [ ]:
# ── Example A: optimise for fastest route (all modes) ─────────────────────────
if ORIGIN_LOC and DEST_LOC:
    route_fast = plan_route(
        origin_location_id=ORIGIN_LOC,
        destination_location_id=DEST_LOC,
        departure_datetime="2025-07-16T09:00:00",
        modes=None,           # all available modes
        optimize_for="time",
        session_id=SESSION_ID,
    )
    print(f"{route_fast['origin_name']} → {route_fast['destination_name']}")
    for r in route_fast["routes"][:3]:
        print(f"  {r['mode']:10s}  {r['duration_min']} min  ${r.get('cost', 0):.2f}")

In [ ]:
# ── Example B: optimise for cheapest walking + bus only ──────────────────────
if ORIGIN_LOC and DEST_LOC:
    route_cheap = plan_route(
        origin_location_id=ORIGIN_LOC,
        destination_location_id=DEST_LOC,
        departure_datetime="2025-07-16T09:00:00",
        modes="walking,bus",
        optimize_for="cost",
        session_id=SESSION_ID,
    )
    for r in route_cheap["routes"]:
        print(f"  {r['mode']:10s}  {r['duration_min']} min  ${r.get('cost', 0):.2f}")

In [ ]:
# ── Example C: compare all modes side-by-side ────────────────────────────────
if ORIGIN_LOC and DEST_LOC:
    comparison = compare_transport_modes(
        origin_location_id=ORIGIN_LOC,
        destination_location_id=DEST_LOC,
        departure_datetime="2025-07-16T09:00:00",
    )
    print(f"{'Mode':12s}  {'Min':>5s}  {'Cost':>7s}  Congestion")
    for r in comparison["routes"]:
        print(f"  {r['mode']:10s}  {r['duration_min']:>5}  "
              f"${r.get('cost', 0):>6.2f}  "
              f"{'yes' if r.get('congestion_applied') else 'no'}")

In [ ]:
# ── Example D: what's near the hotel (by lat/lon) ────────────────────────────
if dest_locs:
    anchor = dest_locs[0]  # use first location as anchor
    nearby = proximity_search(
        lat=anchor["lat"],
        lon=anchor["lon"],
        top_n=5,
        sort_by="distance",
        location_type="attraction",
        city_id=DEST_CITY,
        mode="walking",
    )
    print(f"{nearby['count']} nearby attraction(s):")
    for loc in nearby["results"]:
        print(f"  {loc.get('name', loc.get('location_id', '')):30s}  "
              f"{loc.get('distance_km', '?'):.2f} km")

---
## 9 · Weather

Fetch forecasts to help users decide what to pack or when to visit outdoor attractions.

In [ ]:
def get_weather_forecast(
    *,
    city_id: str,
    start_date: str,  # "YYYY-MM-DD"
    end_date: str,    # "YYYY-MM-DD"
) -> list[dict]:
    """
    Daily weather forecast over a date range.

    Returns list of:
        {date, condition, temperature_c, precipitation_mm,
         wind_speed_kmh, visibility_km, humidity_pct}
    """
    return _get(
        "/weather/forecast",
        city_id=city_id,
        start_date=start_date,
        end_date=end_date,
    )

def get_weather_snapshot(
    *,
    city_id: str,
    date: str,  # "YYYY-MM-DD"
) -> dict | None:
    """
    Weather for a single city / date — returns None if no data exists.
    Use when the agent needs to answer "what's the weather on day X".
    """
    return _get("/weather/snapshot", city_id=city_id, date=date)

In [ ]:
# ── Example A: full trip-window forecast ─────────────────────────────────────
forecast = get_weather_forecast(
    city_id=DEST_CITY,
    start_date="2025-07-15",
    end_date="2025-07-22",
)
print(f"{'Date':12s}  {'Condition':15s}  {'Temp °C':>8s}  {'Rain mm':>8s}")
for day in forecast:
    print(f"  {day['date']}  {day['condition']:15s}  "
          f"{day['temperature_c']:>6.1f}°C  "
          f"{day['precipitation_mm']:>6.1f} mm")

In [ ]:
# ── Example B: compare weather across origin and destination on same day ─────
for city, label in [(ORIGIN_CITY, "Origin"), (DEST_CITY, "Destination")]:
    snap = get_weather_snapshot(city_id=city, date="2025-07-16")
    if snap:
        print(f"  {label:12s}  {snap['condition']:15s}  {snap['temperature_c']:.1f}°C  "
              f"{snap['precipitation_mm']:.1f} mm rain")

---
## 10 · Trip Plan

Build up, inspect, and adjust the traveller's trip plan within the session.

In [ ]:
def add_trip_item(
    *,
    session_id: str,
    item_type: str,      # "flight" | "hotel" | "event" | "attraction"
    ref_id: str,         # flight_id / hotel_id / event_id / attraction_id
    date: str = "",      # "YYYY-MM-DD" — the day this item is used
    cost: float = 0.0,
    metadata: dict | None = None,  # arbitrary key/value for display
) -> dict:
    """
    Add a selected item to the trip plan.

    Returns updated plan: {items: list, total_cost}
    """
    return _post(
        f"/session/{session_id}/trip_plan/add",
        item_type=item_type,
        ref_id=ref_id,
        date=date,
        cost=cost,
        metadata=metadata or {},
    )

def get_trip_plan(
    *,
    session_id: str,
) -> dict:
    """Return current trip plan items and running total cost."""
    return _get(f"/session/{session_id}/trip_plan")

def get_trip_plan_summary(
    *,
    session_id: str,
) -> dict:
    """
    Budget summary — total cost, remaining budget, cost breakdown by category.

    Returns:
        {total_cost, budget_total, budget_remaining, cost_by_category, item_count}
    """
    return _get(f"/session/{session_id}/trip_plan/summary")

def remove_trip_item(
    *,
    session_id: str,
    item_id: str,   # from TripPlanItem.item_id
) -> dict:
    """Remove an item from the trip plan by its item_id."""
    return _delete(f"/session/{session_id}/trip_plan/{item_id}")

def reset_trip_plan(
    *,
    session_id: str,
) -> dict:
    """Clear all trip plan items while keeping session preferences intact."""
    return _post(f"/session/{session_id}/reset")

In [ ]:
# ── Example: build a trip plan incrementally then inspect budget ──────────────

# 1. Add a flight
if FLIGHT_ID:
    flight_info = result_all["flights"][0]
    add_trip_item(
        session_id=SESSION_ID,
        item_type="flight",
        ref_id=FLIGHT_ID,
        date="2025-07-15",
        cost=flight_info["total_price"],
        metadata={
            "airline": flight_info["airline"],
            "flight_number": flight_info["flight_number"],
        },
    )

# 2. Add a hotel
if HOTEL_ID:
    hotel_info = hotels_all["hotels"][0]
    add_trip_item(
        session_id=SESSION_ID,
        item_type="hotel",
        ref_id=HOTEL_ID,
        date="2025-07-15",
        cost=hotel_info["price_per_night"] * 7,  # 7 nights
        metadata={"name": hotel_info["name"], "nights": 7},
    )

# 3. Add an event
if EVENT_ID:
    event_info = (events_all or concerts)[0]
    add_trip_item(
        session_id=SESSION_ID,
        item_type="event",
        ref_id=EVENT_ID,
        date=event_info["start_datetime"][:10],
        cost=event_info.get("price_per_ticket", 0) * 2,  # 2 tickets
        metadata={"name": event_info["name"], "tickets": 2},
    )

print("Items added.")

In [ ]:
# ── View the trip plan and budget summary ─────────────────────────────────────
plan = get_trip_plan(session_id=SESSION_ID)
print(f"Trip plan ({len(plan['items'])} item(s))  —  total: ${plan['total_cost']:.2f}")
for item in plan["items"]:
    print(f"  [{item['item_type']:10s}]  {item.get('metadata', {}).get('name', item['ref_id'])[:30]:30s}  "
          f"${item['cost']:.2f}")

summary = get_trip_plan_summary(session_id=SESSION_ID)
print(f"\nBudget: ${summary.get('budget_total', 'N/A')}  "
      f"Remaining: ${summary.get('budget_remaining', 'N/A')}")
print("By category:", summary.get("cost_by_category", {}))

In [ ]:
# ── Remove the last item, then confirm the plan updated ──────────────────────
if plan["items"]:
    last_id = plan["items"][-1]["item_id"]
    updated = remove_trip_item(session_id=SESSION_ID, item_id=last_id)
    print(f"After removal: {len(updated['items'])} item(s)  total ${updated['total_cost']:.2f}")

---
## 11 · End-to-End Snapshot

A single call that gives the agent everything it needs to resume or audit a session.

In [ ]:
# Full session state — call this to bootstrap the agent at the start of each turn
state = get_session(session_id=SESSION_ID)

print("world_id  :", state["world_id"])
print("prefs     :", state["preferences"])
print("plan items:", len(state["trip_plan"]["items"]))
print("chat msgs :", len(state["chat_history"]))
print("llm_conn  :", state["llm_connected"])

In [ ]:
# Interaction log — download for offline evaluation of agent quality
log = get_interaction_log(session_id=SESSION_ID)
print(f"{len(log)} interaction(s) logged")
if log:
    pp.pprint(log[-1])  # show most recent entry

In [ ]:
# Cleanup — close the shared httpx client when done
client.close()
print("Done.")